# SQL Connectivity Diagnostics
This notebook tests private connectivity from a Fabric notebook runtime to Azure SQL and records detailed troubleshooting output.

Run order:
1. Review runtime context.
2. Confirm connection settings.
3. Check DNS and TCP reachability.
4. Attempt a database connection.
5. Run validation queries.
6. Review the persisted diagnostics record.

Security notes:
- Do not hardcode passwords, secrets, or tokens.
- Prefer environment variables, notebook parameters, or Key Vault-backed secret retrieval.
- Keep `DEMO_MODE = True` until settings are verified.

In [ ]:
import json
import os
import platform
import socket
import sys
import time
import traceback
from datetime import datetime, timezone

DEMO_MODE = True
WORKSPACE_ID = os.getenv("FABRIC_WORKSPACE_ID", "795ce5db-7ea0-4a7c-ba64-e27c9fb568f4")
DEFAULT_RESULTS_PATH = "/lakehouse/default/Files/sql_connectivity_diagnostics"

RUN_CONTEXT = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "hostname": socket.gethostname(),
    "workspace_id": WORKSPACE_ID,
    "demo_mode": DEMO_MODE,
}

try:
    RUN_CONTEXT["spark_available"] = spark is not None
except Exception:
    RUN_CONTEXT["spark_available"] = False

interesting_env = [
    "FABRIC_WORKSPACE_ID",
    "SQL_SERVER_NAME",
    "SQL_DATABASE_NAME",
    "SQL_PORT",
    "SQL_AUTH_MODE",
    "SQL_USERNAME",
    "SQL_PASSWORD_SECRET_NAME",
    "SQL_TARGET_QUERY",
]
RUN_CONTEXT["environment"] = {name: os.getenv(name) for name in interesting_env if os.getenv(name)}

TEST_RECORD = {
    "run_context": RUN_CONTEXT,
    "connection_settings": {},
    "dns_check": {},
    "tcp_check": {},
    "connection_attempt": {},
    "queries": [],
    "errors": [],
}

print(json.dumps(RUN_CONTEXT, indent=2, default=str))

## 1. Capture Runtime and Workspace Context
Collect notebook runtime details such as Python version, Spark availability, hostname, timestamp, and selected environment variables to help correlate failures with the execution environment.

## 2. Load Connection Settings Securely
Read SQL server name, database name, port, authentication mode, and secret references from environment variables or parameters without hardcoding credentials.

In [ ]:
def load_connection_settings():
    settings = {
        "server": os.getenv("SQL_SERVER_NAME", "sqlserver-sk2.database.windows.net"),
        "database": os.getenv("SQL_DATABASE_NAME", "sqldemo"),
        "port": int(os.getenv("SQL_PORT", "1433")),
        "auth_mode": os.getenv("SQL_AUTH_MODE", "access_token"),
        "username": os.getenv("SQL_USERNAME"),
        "password_secret_name": os.getenv("SQL_PASSWORD_SECRET_NAME"),
        "minimal_query": "SELECT 1 AS connectivity_check",
        "target_query": os.getenv("SQL_TARGET_QUERY", "SELECT TOP 5 name FROM sys.tables ORDER BY name"),
        "results_path": os.getenv("SQL_DIAGNOSTICS_PATH", DEFAULT_RESULTS_PATH),
    }
    TEST_RECORD["connection_settings"] = {
        key: value for key, value in settings.items()
        if key not in {"username", "password_secret_name"}
    }
    return settings

settings = load_connection_settings()
print(json.dumps(TEST_RECORD["connection_settings"], indent=2))

## 3. Validate DNS and TCP Reachability
Resolve the SQL host and test whether the target TCP port is reachable before attempting a database login.

In [ ]:
dns_start = time.perf_counter()
try:
    addrinfo = socket.getaddrinfo(settings["server"], settings["port"], proto=socket.IPPROTO_TCP)
    addresses = sorted({entry[4][0] for entry in addrinfo})
    TEST_RECORD["dns_check"] = {
        "server": settings["server"],
        "resolved_addresses": addresses,
        "duration_ms": round((time.perf_counter() - dns_start) * 1000, 2),
    }
except Exception as exc:
    TEST_RECORD["dns_check"] = {
        "server": settings["server"],
        "duration_ms": round((time.perf_counter() - dns_start) * 1000, 2),
        "error": str(exc),
    }
    TEST_RECORD["errors"].append({
        "stage": "dns_resolution",
        "type": type(exc).__name__,
        "message": str(exc),
        "traceback": traceback.format_exc(),
    })

port_start = time.perf_counter()
try:
    with socket.create_connection((settings["server"], settings["port"]), timeout=10):
        TEST_RECORD["tcp_check"] = {
            "reachable": True,
            "duration_ms": round((time.perf_counter() - port_start) * 1000, 2),
        }
except Exception as exc:
    TEST_RECORD["tcp_check"] = {
        "reachable": False,
        "duration_ms": round((time.perf_counter() - port_start) * 1000, 2),
        "error": str(exc),
    }
    TEST_RECORD["errors"].append({
        "stage": "tcp_connect",
        "type": type(exc).__name__,
        "message": str(exc),
        "traceback": traceback.format_exc(),
    })

print(json.dumps({"dns_check": TEST_RECORD["dns_check"], "tcp_check": TEST_RECORD["tcp_check"]}, indent=2))

## 4. Create a SQL Connection
Build a SQL client connection, open the session, and measure connection latency while capturing driver and authentication diagnostics.

In [ ]:
connection = None
cursor = None
pyodbc = None
connection_error = None

try:
    import pyodbc
    TEST_RECORD["connection_attempt"]["client"] = "pyodbc"
except Exception as exc:
    TEST_RECORD["connection_attempt"]["client"] = "pyodbc_unavailable"
    TEST_RECORD["errors"].append({
        "stage": "driver_import",
        "type": type(exc).__name__,
        "message": str(exc),
        "traceback": traceback.format_exc(),
    })

if not DEMO_MODE and pyodbc is not None:
    conn_parts = [
        "Driver={ODBC Driver 18 for SQL Server}",
        f"Server=tcp:{settings['server']},{settings['port']}",
        f"Database={settings['database']}",
        "Encrypt=yes",
        "TrustServerCertificate=no",
        "Connection Timeout=30",
    ]

    token = os.getenv("SQL_ACCESS_TOKEN")
    if settings["auth_mode"] == "access_token" and not token:
        try:
            token = mssparkutils.credentials.getToken("https://database.windows.net/")
        except Exception as exc:
            TEST_RECORD["errors"].append({
                "stage": "token_acquisition",
                "type": type(exc).__name__,
                "message": str(exc),
                "traceback": traceback.format_exc(),
            })

    start = time.perf_counter()
    try:
        if settings["auth_mode"] == "access_token" and token:
            import struct
            token_bytes = token.encode("utf-16-le")
            token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
            SQL_COPT_SS_ACCESS_TOKEN = 1256
            connection = pyodbc.connect(";".join(conn_parts), attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})
        else:
            username = os.getenv("SQL_USERNAME")
            password = os.getenv("SQL_PASSWORD")
            if username and password:
                conn_parts.extend([f"UID={username}", f"PWD={password}"])
                connection = pyodbc.connect(";".join(conn_parts))
            else:
                raise RuntimeError("No supported authentication material was available.")

        TEST_RECORD["connection_attempt"].update({
            "status": "connected",
            "duration_ms": round((time.perf_counter() - start) * 1000, 2),
            "auth_mode": settings["auth_mode"],
        })
        cursor = connection.cursor()
    except Exception as exc:
        connection_error = exc
        TEST_RECORD["connection_attempt"].update({
            "status": "failed",
            "duration_ms": round((time.perf_counter() - start) * 1000, 2),
            "auth_mode": settings["auth_mode"],
            "message": str(exc),
        })
        TEST_RECORD["errors"].append({
            "stage": "sql_connect",
            "type": type(exc).__name__,
            "message": str(exc),
            "args": [str(arg) for arg in getattr(exc, "args", [])],
            "traceback": traceback.format_exc(),
        })
else:
    TEST_RECORD["connection_attempt"].update({
        "status": "skipped" if DEMO_MODE else "not_attempted",
        "auth_mode": settings["auth_mode"],
    })

print(json.dumps(TEST_RECORD["connection_attempt"], indent=2, default=str))

## 5. Run a Minimal Connectivity Query
Execute a simple `SELECT 1` query to confirm that authentication, session creation, and query execution all succeed.

## 6. Execute a Targeted Validation Query
Run a small query against the intended database or schema and inspect the returned rows to verify that expected objects are accessible.

## 7. Capture Detailed Exception Diagnostics
Wrap connection and query steps in exception handling, collect error messages, stack traces, and timestamps, and store them in a structured diagnostics record.

In [ ]:
def run_query(label, sql_text):
    started = time.perf_counter()
    record = {
        "label": label,
        "sql": sql_text,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    try:
        if DEMO_MODE:
            record["status"] = "skipped_demo_mode"
        elif cursor is None:
            record["status"] = "skipped_no_connection"
        else:
            cursor.execute(sql_text)
            rows = cursor.fetchmany(5)
            record["status"] = "succeeded"
            record["duration_ms"] = round((time.perf_counter() - started) * 1000, 2)
            record["row_count_preview"] = len(rows)
            record["rows"] = [list(row) for row in rows]
    except Exception as exc:
        record["status"] = "failed"
        record["duration_ms"] = round((time.perf_counter() - started) * 1000, 2)
        record["message"] = str(exc)
        TEST_RECORD["errors"].append({
            "stage": f"query:{label}",
            "type": type(exc).__name__,
            "message": str(exc),
            "args": [str(arg) for arg in getattr(exc, "args", [])],
            "traceback": traceback.format_exc(),
        })
    TEST_RECORD["queries"].append(record)
    return record

minimal_result = run_query("minimal_connectivity", settings["minimal_query"])
target_result = run_query("target_validation", settings["target_query"])

print(json.dumps({"minimal": minimal_result, "target": target_result}, indent=2, default=str))

## 8. Persist Test Results for Support Review
Assemble the diagnostics into a structured JSON record, display a summary in the notebook, and attempt to write the output to the default lakehouse Files area for support escalation.

In [ ]:
TEST_RECORD["completed_utc"] = datetime.now(timezone.utc).isoformat()
print(json.dumps(TEST_RECORD, indent=2, default=str))

if cursor is not None:
    cursor.close()
if connection is not None:
    connection.close()

if not DEMO_MODE:
    try:
        os.makedirs(settings["results_path"], exist_ok=True)
        output_path = os.path.join(
            settings["results_path"],
            f"sql_connectivity_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json",
        )
        with open(output_path, "w", encoding="utf-8") as handle:
            json.dump(TEST_RECORD, handle, indent=2, default=str)
        print(f"Diagnostics written to {output_path}")
    except Exception as exc:
        TEST_RECORD["errors"].append({
            "stage": "persist_results",
            "type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(),
        })
        print("Diagnostics persistence failed:")
        print(str(exc))